## LOAD DATA

In [ ]:
from fastai.vision.all import *
from fastai.vision.gan import *




In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from PIL.ImageFilter import MedianFilter
from PIL.Image import BICUBIC
from PIL import Image,ImageOps
from skimage.io import imread, imshow, imsave
from skimage import io
import skimage
import torchvision.transforms as transforms
from pydicom import dcmread
import segmentation_models_pytorch as smp
from collections import namedtuple
import nibabel as nib
import imageio
from torchvision.transforms import CenterCrop
from monai.losses.ssim_loss import SSIMLoss #Valid for 3D
from monai.losses import MultiScaleLoss
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import cv2
#import visvis as vv

In [ ]:
# @patch
# def predict(self:Learner, item, rm_type_tfms=None, with_input=False):
#     print('Line 1')
#     dl = self.dls.test_dl([item], rm_type_tfms=rm_type_tfms, num_workers=0)
#     print('Line 2')
#     inp,preds,_,dec_preds = self.get_preds(dl=dl, with_input=True, with_decoded=True)
#     print('Line 3')
#     i = getattr(self.dls, 'n_inp', -1)
#     print('Line 4')
#     inp = (inp,) if i==1 else tuplify(inp)
#     print('Line 5')
#     dec = self.dls.decode_batch(inp + tuplify(dec_preds))[0]
#     print('Line 6')
#     dec_inp,dec_targ = map(detuplify, [dec[:i],dec[i:]])
#     print('Line 7')
#     res = dec_targ,dec_preds[0],preds[0]
#     print('Line 8')
#     if with_input: res = (dec_inp,) + res
#     return res

In [ ]:
#SPECIFY A LOCAL PATH, PATHS TO NAS ARE NOT VALID.
path=Path(r"DATA_ROOT\3_StructuralPreProcessing\volume_slices_all_2\FLAIR_3D_SPIR_TRA_IQ")
path_label=Path(r"DATA_ROOT\3_StructuralPreProcessing\volume_slices_all_2\T1W_CE_3D_FFE_TRA_IQ")
files_im=get_files(path,'.gz')
pct=1
files=files_im[0:round(len(files_im)*pct)]
files

In [ ]:
def LoadMod(mod):
    if mod==1:
        mod_name='FLAIR_3D_SPIR_TRA_IQ'
    elif mod==2:
        mod_name='SWIm'
    elif mod==3:
        mod_name='T1W_3D_FFE_TRA_IQ'
    elif mod==4:
        mod_name='T1W_CE_3D_FFE_TRA_IQ'#'T1W_SUB_PYTHON'#'T1W_CE_3D_FFE_TRA_IQ'
    else:
        print('Unrecognized Modality_')
    def loadImage(o):
        path=str(o)
        new_path=Path(re.sub("FLAIR_3D_SPIR_TRA_IQ",mod_name,path))#Regular expressions
        im=nib.load(new_path)
        im=im.get_fdata()
        if ((np.max(im)-np.min(im))!=0):#In case we have empty images!!
            im=(im-np.min(im))/(np.max(im)-np.min(im))*255#Normalize
        
        pilIm=PILImage.create(im.astype('int'))
        return pilIm
    return loadImage
        

In [ ]:
#CHECK THAT THE FUNCTION IS WORKING
loadTfm=LoadMod(4)
im=loadTfm(files[70])
im.show(cmap='gray')


In [ ]:
#GENERATE DATASET
tfms=[[LoadMod(1)],[LoadMod(2)],[LoadMod(3)],[LoadMod(4)]]
splits=GrandparentSplitter(train_name='Train', valid_name='Valid')(files)
pct=0.5
splits=list(splits)
splits[0]=splits[0][0:round(len(splits[0])*pct)]
splits[1]=splits[1][0:round(len(splits[1])*pct)]


In [ ]:
dsets_gen=Datasets(files,tfms,splits=splits)
print(np.shape(dsets_gen[14][0]))


Nota: los datos tienen que estar estructurados de la siguiente forma:
- FLAIR_3D_SPIR_TRA_IQ

    ---Train
        --- PAT_1
        --- PAT_2
        etc
    ---Valid
        --- PAT_3
        --- PAT_4
    etc
        
        
- SWIm

 etc
 
 Se recomienda hacer una partición de los datos de tal forma que el 80% de los pacientes estén en Train. 

In [ ]:
#GENERATE DATALOADER
class CropPad(Transform):
    def __init__(self,size):self.resize=size
    def encodes(self,x:PILImage):
        x=x.crop_pad(self.resize,pad_mode='zeros') #https://docs.fast.ai/vision.augment.html
        return x

def get_dls(bs,size,resFlag=2):
    if resFlag==1:
        ResTransform=Resize(size)
    elif resFlag==0:
        ResTransform=RandomCrop(size)
    elif resFlag==2:
        ResTransform=CropPad(size)
        
    dls_gen=dsets_gen.dataloaders(bs=bs,after_item=[ResTransform,ToTensor],after_batch=[IntToFloatTensor,Rotate(max_deg=10, p=0.5), FlipItem(p=0.5)],shuffle=True)#El after item y el IntToFloatTensor te lo aplica al valid dset también, pero el data augmentation no.
    dls_gen.c = 3 #Input channels
    return dls_gen

In [ ]:
dls_gen=get_dls(10,[288,224],2) #size=Largest
dls_gen.valid.show_batch(max_n=3,cmap='gray')

In [ ]:
b=dls_gen.one_batch()
b[0].shape

In [ ]:
from skimage.io import imsave
def Mask(img1,img2):
    gamma=0.62 #definir valor en ImgaeJ
    mask=img2>gamma #comprobar tipo. si vector, hacer reshpae
    print(type(mask))
    imshow(mask, cmap='gray')
    
    path_mock=r'DATA_ROOT\3_StructuralPreProcessing\mask'
    im = Image.fromarray(mask)
    imsave(os.path.join(path_mock, 'your_file.jpeg'), mask)

    
    #nib.save(final_img, os.path.join("D:/Volumes convertidos LIDC", 'test4d.nii.gz'))
   
    img1_seg=img1*mask
    #imshow(img1_seg)
    img2_seg=img2*mask
    #imshow(img2_seg)

img1_path=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices\volume_slices\T1W_CE_3D_FFE_TRA_IQ\Train\PAT_3_\PAT_3_T1W_CE_3D_FFE_TRA_IQ_slice_68.nii.gz'
img2_path=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices\volume_slices\FLAIR_3D_SPIR_TRA_IQ\Train\PAT_3_\PAT_3_FLAIR_3D_SPIR_TRA_IQ_slice_68.nii.gz'
#Load images
img1=nib.load(img1_path)
img1=img1.get_fdata()
if ((np.max(img1)-np.min(img1))!=0):#In case we have empty images!!
    img1=(img1-np.min(img1))/(np.max(img1)-np.min(img1))*255#Normalize

pilImg1=PILImage.create(img1.astype('int'))

img2=nib.load(img2_path)
img2=img2.get_fdata()
if ((np.max(img2)-np.min(img2))!=0):#In case we have empty images!!
    img2=(img2-np.min(img2))/(np.max(img2)-np.min(img2))#Normalize

pilImg2=PILImage.create(img2.astype('int'))
mask_pat3_68=Mask(img1,img2)
if os.path.isfile(img2_path):
    filename_zero, extension = os.path.splitext(img2_path)
print(filename_zero)




In [ ]:
#Import masks
mask_path=

## LOSS FUNCTION and METRICS

In [ ]:
#Loss Functions
import pytorch_ssim
from pytorch_msssim import ssim, ms_ssim, SSIM, MS_SSIM #If you need to calculate MS-SSIM/SSIM on normalized images, please denormalize them to the range of [0, 1] or [0, 255] first. https://pypi.org/project/pytorch-msssim/
from torch.nn import MSELoss
from torchmetrics import PeakSignalNoiseRatio
#Img1 is the prediction and img2 is the target

class MS_SSIM_Loss(MS_SSIM):
    def forward(self, img1, img2):
        return (1 - super(MS_SSIM_Loss, self).forward(img1, img2) )
    
class MS_SSIM_Local_Loss(MS_SSIM):
    def forward(self, img1, img2, mask):
        gamma=0.4
        mask=img2>gamma
        img1_seg=img1*mask
        img2_seg=img2*mask
        return (1 - super(MS_SSIM_Local_Loss, self).forward(img1_seg, img2_seg))

class MS_SSIM_Metric(MS_SSIM):
    def forward(self, img1, img2):
        return 100*(super(MS_SSIM_Metric, self).forward(img1, img2) )#it is a percentage

class PSNR_Metric(torch.nn.Module):#It must inherit from Module so we can call forward method.
    def forward(self, img1, img2):
        return 10*torch.log10(img1.max().unsqueeze(0)**2/MSELossFlat()(img1,img2))
    
mssim_metric=MS_SSIM_Metric(data_range=1.0, size_average=True, channel=1)
psnr_metric= PSNR_Metric()




In [ ]:
#COMBINE LOSSES IF NECESSARY
lamb_msssim=1
lambda_mse=0
lambda_mae=1
lambda_local=1
def CombinedLoss(img1,img2):
    criterion_msssim=MS_SSIM_Loss(data_range=1.0, size_average=True, channel=1)
    criterion_mse=MSELossFlat(reduction='mean')
    criterion_mae=nn.L1Loss(reduction='mean')
    criterion_local=MS_SSIM_Local_Loss(data_range=1.0, size_average=True, channel=1)
    return lamb_msssim*criterion_msssim(img1,img2)+lambda_mse*criterion_mse(img1,img2)+lambda_mae*criterion_mae(img1,img2)+lambda_local*criterion_local(img1,img2)#As MSE becomes smaller, SSIM becomes more important. We start by brute force, and we finish by refining perceptual content.

loss_gen=CombinedLoss


## MODEL

In [ ]:
class ShowImageCallback(Callback):
    "Show images"

    def after_epoch(self):
        print('Predicting...')
        full_dec,dec,_=self.predict(files[0])
        print('Showing...')
        full_dec.show()

In [ ]:
smp_flag=1

# CONCATENATE MODALITIES BEFORE FEEDING THEM TO THE MODEL. APPLY SIGMOID TO CLIP OUTPUTS.
class FinalModel(nn.Module):
    def __init__(self,model):
        super().__init__()
        self.model=model
        self.sigmoid=nn.Sigmoid()
    def forward(self,x1,x2,x3):
        x=torch.cat((x1,x2,x3),dim=1)
        x=self.model(x)
        #if type(x) is tuple:#In case dropout is selected for SMP. x[1] corresponds to classficiation scores
            #x=x[0]
        x=self.sigmoid(x)

        return x

#CHOOSE MODEL FROM FASTAI OR SMP.
    
def create_gen_learner():#It mustn´t be pretrained or fastai will automatically add 3 channels to the batches, see https://forums.fast.ai/t/unet-size-error-for-a-custom-dataset/52478/28
    callbacks=[ShowGraphCallback()]# CSVLogger(fname='model.csv', append=False)] #EarlyStoppingCallback(monitor='valid_loss', min_delta=0, patience=15,comp=np.less)]
    if smp_flag==0:
        bbone = resnet50
        FLearner=unet_learner(dls_gen, bbone, loss_func=loss_gen,cbs=callbacks,metrics=psnr_metric,n_out=1,n_in=1,pretrained=False,y_range=[0,1])#.to_fp16()#y_range was added after gen-20220712b
        print('FASTAI UNET selected')
    else:
    #https://smp.readthedocs.io/en/latest/models.html#unet
        model = smp.Unet(
        encoder_name="resnet50",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
        encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=3,                  #  If you use pretrained weights from imagenet - weights of first convolution will be reused. For 1-channel case it would be a sum of weights of first convolution layer
        classes=1)#Number of output channels
        #aux_params={'classes':1,'dropout':0.5}) #Dropoout adds a classification layer on which apply dropout                    # model output channels (number of classes in your dataset)
        bbone=FinalModel(model)
        FLearner= Learner(dls_gen, bbone, loss_func=loss_gen,cbs=callbacks,metrics=psnr_metric)
        print('SMP learner selected')
    return FLearner,bbone

learn_gen,model = create_gen_learner()
#To retrain the network:
#old_model_path=r'DATA_ROOT\models\MRISyth_Comb_res50-05072023_bestEpoch'
#learn_gen.load(old_model_path).to(device)
#print('Model loaded succesfully')


In [ ]:
#CHECK AVAILABLE GPU SPACE. RESTART KERNEL IF NECESSARY
# nvmlInit()
# h = nvmlDeviceGetHandleByIndex(0)
# info = nvmlDeviceGetMemoryInfo(h)
# print(f'total    : {round(info.total/10**9,2)}')
# print(f'free     : {round(info.free/10**9,2)}')
# print(f'used     : {round(info.used/10**9,2)}')

In [ ]:
wd=1e-3
load_flag=0
#name='MRISyth_MSSIM_res50-25082023_bestEpoch'
name='MRISyth_Comb_res50-31082023' #Change name according to date!!

if load_flag:
    mod_path=Path(name)
    print(mod_path)
    learn_gen.load(mod_path).to(device)
    print('Model loaded succesfully')
    
else:
    learn_gen.dls=get_dls(10,[288,224],2)#Pad_Crop. Select the size of the largest patient, and take the closest multiple of 32 using ceil.
    #FIND LEARNING RATE. RUN JUST ONCE.
    #learn_gen.lr_find()
    lr=0.0017 #Value found by lr_find()
    learn_gen.add_cb(SaveModelCallback(fname=name+'_bestEpoch',comp=np.less,with_opt=True)) #Save model at the best valid loss.
    #learn_gen.add_cb(CSVLogger(fname='model.csv', append=False))
    learn_gen.fit_one_cycle(60, lr, wd=wd)


## VALIDATE WITH 3D VOLUME

In [ ]:
save_flag=1
#name='MRISyth_MSSIM_res50-25082023_bestEpoch'
name=name+'_bestEpoch'

if save_flag:
    excel_path=Path(r"DATA_ROOT\4_DL")/name
    if not Path(excel_path).is_dir(): Path.mkdir(excel_path)
else:
    print('Images won´t be saved')

mod_path=Path(name)
# learn_gen.load(mod_path).to(device)
model=learn_gen.model

profiles={'label':[],'network':[]}
metrics={'SSIM':[],'psnr':[]}

pats = [o.name for o in os.scandir(path/'Valid')]# if o.is_file()]
for pat in progress_bar(pats):
    slices_path=get_files(path/'Valid'/pat)
    slice_org=nib.load(slices_path[0])
    org_size=np.shape(slice_org.get_fdata())
    
    full_dec,dec,out=learn_gen.predict(slices_path[0])
    vol_preds = torch.empty((len(slices_path),full_dec.shape[1],full_dec.shape[2]), dtype=torch.float32)
    vol_targs= torch.empty((len(slices_path),org_size[0],org_size[1]), dtype=torch.float32)
    
    loadTfm=LoadMod(4)
    #Obtain predictions
    for i,axial_path in enumerate(slices_path):
        new_path=Path(re.sub("slice_\d*","slice_"+str(i+1),str(axial_path)))
        full_dec,dec,out=learn_gen.predict(new_path)
        del dec,out
        targs=loadTfm(new_path)
        vol_preds[i,:,:]=full_dec
        vol_targs[i,:,:]=ToTensor.encodes(targs)
        
    
    #Convert to volume
    vol_preds=vol_preds.to(torch.float32)
    vol_preds=CenterCrop(org_size)(vol_preds)#Return to original size, because ResTransform  of learn_gen applies a crop_pad.
    #vv.volshow(vol_preds)
    
    if save_flag:
        #Save as tiff
        imageio.volwrite(str(excel_path/pat)+'.tiff',vol_preds)
     

        #Save as Nifti
        vol_nifti=nib.Nifti1Image(vol_preds.numpy().transpose(1,2,0),slice_org.affine,slice_org.header)
        
        nib.save(vol_nifti,str(excel_path/pat)+'.nii.gz')

        #Save profiles
        line=np.round((vol_preds.shape[2])/2).astype('int')
        profiles['label']=vol_preds[np.round(vol_preds.shape[0]/2).astype('int'),:,line]
        profiles['network']=vol_targs[np.round(vol_preds.shape[0]/2).astype('int'),:,line]
        #df=pd.DataFrame(profiles).to_excel(str(excel_path/pat)+'.xlsx')
        
    #Show one 2D image per patient, always the same one to have reproducible outputs
    with plt.style.context('bmh'):#http://www.futurile.net/2016/02/27/matplotlib-beautiful-plots-with-style/
        fig, axes = plt.subplots(1,3,figsize=(20,5))

        ax = axes[0]
        ax.imshow(vol_targs[np.round(vol_targs.shape[0]/2).astype('int'),:,:],cmap='gray')
        ax.set_title('Label_'+pat, fontsize=15)
        ax. axis('off')
        ax = axes[1]
        ax.imshow(vol_preds[np.round(vol_preds.shape[0]/2).astype('int'),:,:],cmap='gray')
        ax.set_title('Predictions_'+pat, fontsize=15)
        ax. axis('off')

        ax = axes[2]
        ax.plot(np.squeeze(vol_preds[np.round(vol_preds.shape[0]/2).astype('int'),np.round(vol_preds.shape[1]/2).astype('int'),:]),color='red',label='Predictions')
        ax.plot(np.squeeze(vol_targs[np.round(vol_targs.shape[0]/2).astype('int'),np.round(vol_targs.shape[1]/2).astype('int'),:]),color='blue',label='Label')
        ax.set_xlabel('pixel position', fontsize=12)
        ax.set_ylabel('Normalized intensity', fontsize=12)
        ax.legend()
     
    #Compute metrics
    vol_preds=IntToFloatTensor.encodes(vol_preds[np.newaxis,np.newaxis,:,:,:])#(N,C,D,W,H)
    vol_targs=IntToFloatTensor.encodes(vol_targs[np.newaxis,np.newaxis,:,:,:])
    ssim_3Dmetric=SSIMLoss(spatial_dims=3)
    
    #metrics['SSIM'].append(100*(1-ssim_3Dmetric(vol_preds,vol_targs,data_range=vol_preds.max().unsqueeze(0))).numpy())
    metrics['SSIM'].append(100*(1-ssim_3Dmetric(vol_preds,vol_targs)).numpy())
    metrics['psnr'].append(psnr_metric(vol_preds,vol_targs).numpy())
    
    del vol_preds,vol_targs
    print(f'Patient {pat} completed')
   


In [ ]:
df_metrics=pd.DataFrame(metrics)
res={'mean':0,'std':0}
res['mean']=df_metrics.mean()
res['std']=df_metrics.std()


pos=int([ind for ind,value in enumerate(metrics['SSIM']) if value==sorted(metrics['SSIM'])[0]][0])
best=int([ind for ind,value in enumerate(metrics['SSIM']) if value==sorted(metrics['SSIM'])[7]][0])
print(metrics['SSIM'])
print(sorted(metrics['SSIM'])[7])

print(sorted(metrics['SSIM']))
print(sorted(metrics['PSNR']))
print(pos)
n_validation=len(pats)

print(f'Worst result for {[string for ind,string in enumerate(pats) if ind==pos][0]}')
print(f'Best result for {[string for ind,string in enumerate(pats) if ind==best][0]}')
pd.DataFrame(res).head()